# Modelagem dos Dados

Esta etapa visa a criar uma estrutura de banco de dados no modelo estrela que possa ser importadada para a ferramenta de análise de dados. 

Há uma limitação que restringe o tamanho máximo de arquivos a serem importados pelo Google Data Studio de 100MB. Como o arquivo de dados concatenados em .csv `BPS_20_26_OrlandoCastro_Atualizado.csv` possui mais de 120MB, não é possível usar esta base de dados concatenada, mesmo que tenha sido previamente tratada.

A modelagem dos dados brutos para um esquema estrela, com a estruturação dos dados em tabela-fato e tabelas-dimensões, além de otimizar o tamanho do dataset, permite a criação de métricas e dimensões que podem ser utilizadas para análises mais complexas.

Apresentamos a seguir a modelagem dos dados em um esquema estrela, com a criação das dimensões e da tabela fato, bem como a exportação das tabelas para arquivos CSV que podem ser importados pelo software de análise de dados.


### 1. Importação dos dados para um dataset usando DuckDb

In [2]:
# Abrir o arquivo BPS_20_26_OrlandoCastro_Atualizado.csv para leitura e inserir os dados em um dataset usando duckdb

import duckdb
import os

# Configurações
pasta_dados = 'dados_tratados'
db_file = os.path.join(pasta_dados, 'dados_bps.db')
csv_entrada = os.path.join(pasta_dados, 'BPS_20_26_OrlandoCastro_atualizado.csv')

con = duckdb.connect(db_file)

# 1. Garantir que a tabela base bps_data existe e está atualizada
con.execute(f"CREATE TABLE IF NOT EXISTS bps_data AS SELECT * FROM read_csv_auto('{csv_entrada}', sep=';');")

print("Tabela bps_data criada com sucesso.")

Tabela bps_data criada com sucesso.


### 2. Criação das Dimensões

Extraído do arquivo `BPS_20_26_OrlandoCastro_Atualizado.csv` as dimensões:
Instituições (Dim_insitituicoes), Municípios (Dim_municipios), Materiais (Dim_materiais), Fornecedores (Dim_fornecedor), Fabricantes (Dim_fabricante), Modalidades de compras (Dim_modalida_compra) e Tipos de Compras (Dim_tipo_compra).

In [3]:
# 2. Criação das Dimensões
# Dim_instituicoes
con.execute("""
CREATE OR REPLACE TABLE Dim_instituicoes AS 
SELECT 
    row_number() OVER () as cod_instituicao, 
    nome_instituicao, 
    esfera, 
    cnpj_instituicao 
FROM (SELECT DISTINCT nome_instituicao, esfera, cnpj_instituicao FROM bps_data ORDER BY nome_instituicao);
""")

# Dim_municipios
con.execute("""
CREATE OR REPLACE TABLE Dim_municipios AS 
SELECT 
    row_number() OVER () as cod_municipio, 
    municipio_instituicao, 
    uf 
FROM (SELECT DISTINCT municipio_instituicao, uf FROM bps_data);
""")

# Dim_materiais (Indexada por codigo_br)
con.execute("""
CREATE OR REPLACE TABLE Dim_materiais AS 
SELECT DISTINCT 
    codigo_br, 
    descricao_catmat 
FROM bps_data
ORDER BY codigo_br;
""")

# Dim_fornecedor
con.execute("""
CREATE OR REPLACE TABLE Dim_fornecedor AS 
SELECT 
    row_number() OVER () as cod_fornecedor, 
    cnpj_fornecedor, 
    fornecedor 
FROM (SELECT DISTINCT cnpj_fornecedor, fornecedor FROM bps_data);
""")

# Dim_fabricante
con.execute("""
CREATE OR REPLACE TABLE Dim_fabricante AS 
SELECT 
    row_number() OVER () as cod_fabricante, 
    cnpj_fabricante, 
    fabricante 
FROM (SELECT DISTINCT cnpj_fabricante, fabricante FROM bps_data);
""")

# Dim_modalidade_compra
con.execute("""
CREATE OR REPLACE TABLE Dim_modalidade_compra AS 
SELECT 
    row_number() OVER () as cod_modalidade, 
    modalidade_compra 
FROM (SELECT DISTINCT modalidade_compra FROM bps_data);
""")

# Dim_tipo_compra
con.execute("""
CREATE OR REPLACE TABLE Dim_tipo_compra AS 
SELECT 
    row_number() OVER () as cod_tipo, 
    tipo_compra 
FROM (SELECT DISTINCT tipo_compra FROM bps_data);
""")

print("Dimensões criadas com sucesso.")

Dimensões criadas com sucesso.


### 3. Criação da tabela fato

A tabela fato será criada a partir do arquivo `BPS_20_26_OrlandoCastro_Atualizado.csv` e terá como chaves estrangeiras as dimensões criadas anteriormente. A tabela fato conterá informações sobre as compras realizadas, incluindo valores, datas e outras métricas relevantes.

In [4]:
# 3. Criação da Tabela Fato
# Join com as dimensões para obter as chaves
con.execute("""
CREATE OR REPLACE TABLE Fato_BPS_20_2026 AS 
SELECT 
    row_number() OVER () as cod_compra, 
    b.ano_compra,
    b.compra as data_compra, 
    i.cod_instituicao, 
    m.cod_municipio, 
    b.codigo_br, 
    f.cod_fornecedor, 
    fab.cod_fabricante,
    mc.cod_modalidade,
    tc.cod_tipo,
    b.unidade_fornecimento, 
    b.capacidade, 
    b.unidade_medida, 
    b.unidade_fornecimento_capacidade,  
    b.qtd_itens_comprados, 
    b.preco_unitario, 
    b.preco_total
FROM bps_data b
JOIN Dim_instituicoes i ON b.cnpj_instituicao = i.cnpj_instituicao AND b.nome_instituicao = i.nome_instituicao
JOIN Dim_municipios m ON b.municipio_instituicao = m.municipio_instituicao AND b.uf = m.uf
JOIN Dim_fornecedor f ON b.cnpj_fornecedor = f.cnpj_fornecedor
JOIN Dim_fabricante fab ON b.cnpj_fabricante = fab.cnpj_fabricante
JOIN Dim_modalidade_compra mc ON b.modalidade_compra = mc.modalidade_compra
JOIN Dim_tipo_compra tc ON b.tipo_compra = tc.tipo_compra;
""")

print("Tabela fato criada com sucesso.")

Tabela fato criada com sucesso.


### 4. Exportação das tabelas para arquivos CSV

Todas as tabelas criadas (dimensões e tabela fato) serão exportadas para arquivos CSV, que poderão ser importados para o Google Data Studio. Cada tabela será salva em um arquivo separado, garantindo que o tamanho de cada arquivo não ultrapasse o limite de 100MB imposto pelo Google Data Studio.

In [5]:
# 4. Exportação para CSV
tabelas = [
    'Fato_BPS_20_2026', 'Dim_instituicoes', 'Dim_municipios', 
    'Dim_materiais', 'Dim_fornecedor', 'Dim_fabricante', 
    'Dim_modalidade_compra', 'Dim_tipo_compra'
]

for tabela in tabelas:
    output_path = os.path.join(pasta_dados, f"{tabela}.csv")
    con.execute(f"COPY {tabela} TO '{output_path}' (HEADER, DELIMITER ';');")
    print(f"Exportado: {output_path}")

con.close()

print("Exportação das tabelas para o formato .csv concluída com sucesso.")

Exportado: dados_tratados\Fato_BPS_20_2026.csv
Exportado: dados_tratados\Dim_instituicoes.csv
Exportado: dados_tratados\Dim_municipios.csv
Exportado: dados_tratados\Dim_materiais.csv
Exportado: dados_tratados\Dim_fornecedor.csv
Exportado: dados_tratados\Dim_fabricante.csv
Exportado: dados_tratados\Dim_modalidade_compra.csv
Exportado: dados_tratados\Dim_tipo_compra.csv
Exportação das tabelas para o formato .csv concluída com sucesso.
